[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [sqlite3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)

# executemany


## What you will be able to do

Insert, update and delete many rows with one `executemany` call, fed from a list or from a
generator, inside one transaction that saves every row or none of them. Read what `rowcount` reports
afterwards, get back the ids of rows that `executemany` inserted, which `lastrowid` does not give,
and say what happens to the rows before one that fails.


## The idea

### The problem

Loading data is most of what a database's writes are. Every Setup cell in this guide loads a year of
readings, and a real loader goes on doing it: the next day's readings from four stations, a batch of
corrections from a technician, a list of empty readings to delete. Each of those is one statement run
for many rows.

Written as a loop, with a commit after every row, a load that could finish in a moment takes
hundreds of times as long, because every commit waits for the disk. Written with `executemany`, it
is fast, and it hides things a loop would have shown. `lastrowid`, which gave the id of a row
inserted with `execute`, is `None` after `executemany`, or still holds the id of an earlier insert,
so code that trusts it files readings under the wrong station. `rowcount` adds up every row changed,
so one correction that matched nothing disappears inside a total. A row that fails at the 20,000th
insert leaves 19,999 rows behind it, saved or not depending on how the connection handles
transactions. And a list of hours handed over where rows were expected makes every hour a row of its
own characters.

### What executemany is

> **`executemany(sql, parameters)`**, a method of a cursor and of a connection, runs one SQL
> statement that changes rows, an `INSERT`, `UPDATE`, `DELETE` or `REPLACE`, once for every item in
> `parameters`, binding the item's values to the statement's placeholders. `parameters` can be any
> **iterable**: a list of tuples or dictionaries, or a **generator** that produces them one at a
> time. It follows the same transaction rules as `execute`, so under the default control every row
> it changes joins one transaction that `commit` saves. Afterwards the cursor's **`rowcount`** is
> the total number of rows changed, and its **`lastrowid`** is whatever it was before, since
> `executemany` never updates it.

### Why it works that way

- **One statement, run again and again.** `executemany` fetches the prepared statement once, then for
  every row binds new values to it and runs it, all inside one call from Python. A loop of
  `conn.execute` reuses the prepared statement too, from the connection's cache, but pays for a call
  and a new cursor on every row.
- **A commit is the slow part.** SQLite's FAQ explains why: by default a commit waits until the data
  is safely stored on the disk, so that a power cut cannot lose it. A transaction around many rows
  pays that wait once, and a commit after every row pays it every time.
- **Rows are taken as they are needed.** `executemany` asks its iterable for the next row only when
  it is ready to insert it, so a generator can feed it rows that never all exist in memory at once.
- **`lastrowid` belongs to `execute`.** Python's documentation says it changes only after an
  `INSERT` or `REPLACE` run by `execute`, and after `executemany` keeps the value it had, `None` on
  a new cursor. The ids of many rows come from `RETURNING` on inserts run one at a time, or from
  reading the rows back by a column that tells them apart.
- **`rowcount` is a total.** After `executemany` it is the sum of the rows every run of the statement
  changed, which says how many rows changed and never which.
- **A failure stops the rows, not the transaction.** When one row fails, `executemany` raises there.
  The rows before it stay in the open transaction until `rollback` or `commit` decides their fate,
  and on a connection with `autocommit=True` they are already saved.

### Where this shows up

PEP 249, the specification most of Python's database drivers share, defines `executemany`, and
psycopg, in the **asyncpg and psycopg3, Deep Dive** guide, follows it, while asyncpg, which does not
follow PEP 249, has an `executemany` of its own. The
**SQLAlchemy, Deep Dive** guide passes a list of dictionaries to one `execute`, which it runs as an
executemany, and gets ids back for many rows by sending them as `INSERT` statements with many
`VALUES` and `RETURNING`. The **Pandas, Deep Dive** guide's `to_sql` writes a DataFrame to a sqlite3
connection through `executemany`. In this guide, every Setup cell that builds `stations.db` loads its
year of readings with one call.

### What this notebook covers

- `executemany` for a year of readings, from a generator, in one transaction
- A commit for every row against one commit, and `executemany` against a loop of `execute`, timed
- A generator in place of a list, and the memory it saves
- `UPDATE` and `DELETE` with `executemany`, with tuples and with dictionaries
- `rowcount` and `lastrowid` after `execute` and after `executemany`
- Ids for many rows: reading them back, `RETURNING` one row at a time, and one `INSERT` with many
  `VALUES`
- A row that fails partway, and the rows before it
- When to use `executemany`, a loop of `execute` with `RETURNING`, or one `INSERT` with many `VALUES`
- A loader for a day of readings that can run twice, and changes nothing when a correction misses
- Seven errors: a list of strings for rows, a `SELECT`, `RETURNING`, a stale `lastrowid`, a
  correction lost in `rowcount`, half a load saved with `autocommit=True`, and a generator read twice

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import sqlite3

conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE readings (id INTEGER PRIMARY KEY, hour TEXT NOT NULL, celsius REAL)")

one = conn.execute("INSERT INTO readings (hour, celsius) VALUES (?, ?)", ("2026-01-01T00:00", -3.5))
print("execute:     rowcount", one.rowcount, "lastrowid", one.lastrowid)

day = ((f"2026-01-01T{hour:02d}:00", -3.5 + hour / 4) for hour in range(1, 24))
many = conn.executemany("INSERT INTO readings (hour, celsius) VALUES (?, ?)", day)
print("executemany: rowcount", many.rowcount, "lastrowid", many.lastrowid)

conn.commit()
print("saved:", conn.execute("SELECT COUNT(*) FROM readings").fetchone()[0])
conn.close()
```

```
execute:     rowcount 1 lastrowid 1
executemany: rowcount 23 lastrowid None
saved: 24
```

One reading went in with `execute`, and the other 23 hours of the day with one `executemany` call,
fed by a generator. Both joined one transaction that `commit` saved. `rowcount` counted the rows
each call inserted, and `lastrowid` knew the id of the first row, and nothing about the 23 after it.


## Setup

Nine imports, and an empty `stations.db` with the two tables as the **Changing a Schema** notebook
left them, STRICT, with a unique station name, one reading for a station and hour, and `CHECK` rules
on latitude and temperature.

- `sqlite3` holds the database and runs every load
- `math`, `datetime` and `timedelta` make the same year of readings the **Why sqlite3** notebook made
- `time` times the loads, and `tracemalloc` measures the memory they use, in the worked examples
- `csv` and `io` read a day of readings from CSV text, in the worked examples
- `Path` names the scratch folder and the databases in it
- `shutil` removes the scratch folder at the end

`TABLES` is the script that creates the two tables, kept so that the timed loads can create them
again in a database of their own.


In [1]:
import csv
import io
import math
import shutil
import sqlite3
import time
import tracemalloc
from datetime import datetime, timedelta
from pathlib import Path

SCRATCH = Path("scratch")
SCRATCH.mkdir(exist_ok=True)
DATABASE = SCRATCH / "stations.db"
DATABASE.unlink(missing_ok=True)
TIMING = SCRATCH / "timing.db"
STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
LATITUDES = {"Bergen": 60.39, "Oslo": 59.91, "Svalbard": 78.22, "Tromso": 69.65, "Kirkenes": 69.73}
TABLES = """
    CREATE TABLE stations (
        id       INTEGER PRIMARY KEY,
        name     TEXT NOT NULL UNIQUE,
        latitude REAL NOT NULL CONSTRAINT plausible_latitude CHECK (latitude BETWEEN -90 AND 90)
    ) STRICT;
    CREATE TABLE readings (
        id         INTEGER PRIMARY KEY,
        station_id INTEGER NOT NULL REFERENCES stations (id),
        hour       TEXT NOT NULL,
        celsius    REAL CONSTRAINT plausible_celsius CHECK (celsius BETWEEN -90 AND 60),
        UNIQUE (station_id, hour)
    ) STRICT;
"""


def year_of_readings():
    """Every hour of 2025 at the four stations, as Why sqlite3 made them, with Svalbard silent on 2 March."""
    for n in range(365 * 24):
        hour = datetime(2025, 1, 1) + timedelta(hours=n)
        season = -math.cos(2 * math.pi * (n - 400) / (365 * 24))
        day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
        for i, (station, mean) in enumerate(STATIONS.items()):
            if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
                celsius = None
            else:
                wobble = ((n * 37 + i * 101) % 17 - 8) / 10
                celsius = round(mean + 9 * season + 3 * day + wobble, 1) + 0.0
            yield station, hour.strftime("%Y-%m-%dT%H:%M"), celsius


build = sqlite3.connect(DATABASE)
build.executescript(TABLES)
build.close()

print("built", DATABASE)


built scratch/stations.db


## Worked examples

### A year of readings, in one call

`executemany` runs one statement for every item of an iterable. The stations go in from
`LATITUDES.items()`, pairs of a name and a latitude. The readings go in from a generator expression
that puts every station's id in place of its name as the row goes by, with the ids read back from the
table. `with conn:` commits each load when its block ends:


In [2]:
INSERT_READING = "INSERT INTO readings (station_id, hour, celsius) VALUES (?, ?, ?)"

conn = sqlite3.connect(DATABASE)
with conn:
    stations = conn.executemany("INSERT INTO stations (name, latitude) VALUES (?, ?)", LATITUDES.items())
print("stations inserted:", stations.rowcount)

ids = dict(conn.execute("SELECT name, id FROM stations ORDER BY id"))
with conn:
    readings = conn.executemany(
        INSERT_READING,
        ((ids[station], hour, celsius) for station, hour, celsius in year_of_readings()),
    )
print("readings inserted:", readings.rowcount)
print("ids:", ids)


stations inserted: 5
readings inserted: 35040
ids: {'Bergen': 1, 'Oslo': 2, 'Svalbard': 3, 'Tromso': 4, 'Kirkenes': 5}


Two calls loaded five stations and 35,040 readings, and `rowcount` holds how many rows each call
inserted. Under the default transaction control, the first `INSERT` of each call began a transaction,
every other row joined it, and `with conn:` committed once. The ids came from reading the stations
back by name rather than from the inserts, for a reason the section on `lastrowid` shows.

### A commit for every row, or one

Where the transaction ends matters far more than how the rows are sent. `timed_load` creates the
tables again in `timing.db` and inserts rows there, with `executemany` or with a loop of
`conn.execute`, on a connection with `autocommit=True`, which saves every row as it goes, or with
`autocommit=False`, which saves them all at one `commit`. Timings change from machine to machine, so
the cell prints comparisons, with room to spare, instead of seconds:


In [3]:
ROWS = [(ids[station], hour, celsius) for station, hour, celsius in year_of_readings()]


def timed_load(rows, *, autocommit, many):
    """Seconds taken to insert rows into new tables in timing.db, with executemany or a loop of execute."""
    TIMING.unlink(missing_ok=True)
    timing = sqlite3.connect(TIMING, autocommit=autocommit)
    timing.executescript(TABLES)
    started = time.perf_counter()
    if many:
        timing.executemany(INSERT_READING, rows)
    else:
        for row in rows:
            timing.execute(INSERT_READING, row)
    timing.commit()
    seconds = time.perf_counter() - started
    timing.close()
    return seconds


every_row = timed_load(ROWS[:1000], autocommit=True, many=True)
one_commit = timed_load(ROWS[:1000], autocommit=False, many=True)
print("1,000 rows, a commit for every row took more than 50 times as long as one commit:", every_row > 50 * one_commit)

loop = timed_load(ROWS, autocommit=False, many=False)
many = timed_load(ROWS, autocommit=False, many=True)
print("35,040 rows at one commit, a loop of conn.execute took longer than executemany:", loop > many)


1,000 rows, a commit for every row took more than 50 times as long as one commit: True
35,040 rows at one commit, a loop of conn.execute took longer than executemany: True


On the machine this notebook was written on, a commit for every row took several hundred times as
long as one commit, which for a year of readings is the difference between a moment and a long wait.
With `autocommit=True` the `commit` in `timed_load` does nothing, and every row was saved, and
waited for, as it went in. The loop of `conn.execute` took about three times as long as
`executemany`, for the calls and cursors it made. The seconds differ from machine to machine, and
which of each pair is slower does not.

### A generator in place of a list

`executemany` takes the next row only when it is ready to insert it. Here the same year is loaded
from a list built first and from a generator, while `tracemalloc` records the most memory Python held
at any moment:


In [4]:
def peak_memory(load):
    """The most memory, in bytes, that Python's objects used at once while load ran."""
    tracemalloc.start()
    load()
    peak = tracemalloc.get_traced_memory()[1]
    tracemalloc.stop()
    return peak


def rows_of_the_year():
    """The year of readings as rows for the readings table, made one at a time."""
    return ((ids[station], hour, celsius) for station, hour, celsius in year_of_readings())


as_list = peak_memory(lambda: timed_load(list(rows_of_the_year()), autocommit=False, many=True))
as_generator = peak_memory(lambda: timed_load(rows_of_the_year(), autocommit=False, many=True))
print("the rows in a list needed more than 100 times the memory of a generator:", as_list > 100 * as_generator)


the rows in a list needed more than 100 times the memory of a generator: True


The list held all 35,040 rows before the first insert, and the generator held about one row at a
time. On the machine this notebook was written on, that was a few megabytes against a few kilobytes.
For a year of readings a few megabytes do not matter, and for a file of a hundred million lines they
decide whether the load runs at all.

### UPDATE and DELETE, with dictionaries

`executemany` runs an `UPDATE` or a `DELETE` for every item just as it runs an `INSERT`. With named
placeholders, every item is a dictionary. Here three corrections are applied, and Svalbard's 24 empty
readings for 2 March are deleted:


In [5]:
corrections = [
    {"station_id": ids["Oslo"], "hour": "2025-06-01T12:00", "celsius": 14.2},
    {"station_id": ids["Oslo"], "hour": "2025-06-01T13:00", "celsius": 14.6},
    {"station_id": ids["Bergen"], "hour": "2025-06-01T12:00", "celsius": 15.1},
]
with conn:
    corrected = conn.executemany(
        "UPDATE readings SET celsius = :celsius WHERE station_id = :station_id AND hour = :hour", corrections)
print("corrections:", len(corrections), "| rows changed:", corrected.rowcount)

empty_hours = [(ids["Svalbard"], f"2025-03-02T{hour:02d}:00") for hour in range(24)]
with conn:
    removed = conn.executemany("DELETE FROM readings WHERE station_id = ? AND hour = ? AND celsius IS NULL", empty_hours)
print("empty readings deleted:", removed.rowcount)
print("readings left:", conn.execute("SELECT COUNT(*) FROM readings").fetchone()[0])


corrections: 3 | rows changed: 3
empty readings deleted: 24
readings left: 35016


`rowcount` added up the rows every run of the statement changed: three corrections, one row each, and
24 deletions. A dictionary names its values, so their order in it does not matter, and the tuples in
`empty_hours` hold their values in the order of the placeholders.

### rowcount and lastrowid

The same kind of insert through `execute` and through `executemany`, rolled back afterwards:


In [6]:
cursor = conn.execute("INSERT INTO stations (name, latitude) VALUES (?, ?)", ("Bodo", 67.28))
print("execute:     rowcount", cursor.rowcount, "| lastrowid", cursor.lastrowid)

cursor = conn.executemany("INSERT INTO stations (name, latitude) VALUES (?, ?)", [("Alta", 69.97), ("Vardo", 70.37)])
print("executemany: rowcount", cursor.rowcount, "| lastrowid", cursor.lastrowid)

conn.rollback()


execute:     rowcount 1 | lastrowid 6
executemany: rowcount 2 | lastrowid None


Both calls counted their rows, and only `execute` reported an id. `conn.executemany` makes a new
cursor, whose `lastrowid` starts as `None`, and nothing in `executemany` changes it. The rollback
removed the three stations, which the first insert's transaction held.

### Ids for many rows

Three ways to know the ids of several new rows. `executemany` followed by reading the rows back by
their unique name, a loop of `execute` with `RETURNING id`, and one `INSERT` with a `VALUES` list
for every row and `RETURNING id, name`:


In [7]:
with conn:
    conn.executemany("INSERT INTO stations (name, latitude) VALUES (?, ?)", [("Bodo", 67.28), ("Alta", 69.97)])
read_back = dict(conn.execute("SELECT name, id FROM stations WHERE name IN (?, ?) ORDER BY id", ("Bodo", "Alta")))
print("read back by name:       ", read_back)

with conn:
    returned = {name: conn.execute("INSERT INTO stations (name, latitude) VALUES (?, ?) RETURNING id",
                                   (name, latitude)).fetchone()[0]
                for name, latitude in [("Vardo", 70.37), ("Hammerfest", 70.66)]}
print("RETURNING, row by row:   ", returned)

with conn:
    rows = conn.execute("INSERT INTO stations (name, latitude) VALUES (?, ?), (?, ?) RETURNING id, name",
                        ("Narvik", 68.44, "Honningsvag", 70.98)).fetchall()
print("one INSERT, many VALUES: ", {name: station_id for station_id, name in rows})


read back by name:        {'Bodo': 6, 'Alta': 7}
RETURNING, row by row:    {'Vardo': 8, 'Hammerfest': 9}
one INSERT, many VALUES:  {'Narvik': 10, 'Honningsvag': 11}


All three gave every new station its id. Reading back needs a column that tells the rows apart, here
the unique `name`. `RETURNING` with `execute` gives one id at a time, in the order the rows were
sent. A single `INSERT` with many `VALUES` returns every id at once, but SQLite's documentation says
the rows of `RETURNING` come back in no particular order, so the query returns each name with its
id, and a statement can hold only as many placeholders as SQLite allows, a limit that depends on how
SQLite was built.

### A row that fails partway

A day of readings for Kirkenes, whose sensor sent 999 degrees at 15:00, which `plausible_celsius`
refuses. The rows before it are inserted, and wait in the open transaction:


In [8]:
day = [(ids["Kirkenes"], f"2026-01-01T{hour:02d}:00", -8.0 + hour / 4) for hour in range(24)]
day[15] = (ids["Kirkenes"], "2026-01-01T15:00", 999.0)

try:
    conn.executemany(INSERT_READING, day)
except sqlite3.IntegrityError as error:
    print("refused:", error)

waiting = "SELECT COUNT(*) FROM readings WHERE station_id = ? AND hour >= '2026'"
print("in a transaction:", conn.in_transaction)
print("rows waiting in it:", conn.execute(waiting, (ids["Kirkenes"],)).fetchone()[0])
conn.rollback()
print("after rollback:", conn.execute(waiting, (ids["Kirkenes"],)).fetchone()[0])


refused: CHECK constraint failed: plausible_celsius
in a transaction: True
rows waiting in it: 15
after rollback: 0


The first 15 hours went in before `executemany` reached 15:00 and raised, and they stayed in the
transaction. A `commit` now would have saved a day with nine hours missing, and the `rollback`
removed all 15. `with conn:` makes that choice by itself, rolling back when its block raises, which
is the reason every load in this notebook sits inside one.

### executemany, RETURNING row by row, or one INSERT

| Write | When | Why |
|---|---|---|
| `executemany` in one transaction | many rows whose ids you do not need back, or can read back by a unique column | one call runs one statement for every row, a generator can feed it, and one commit saves them |
| a loop of `execute` with `RETURNING`, in one transaction | rows whose ids you need, each matched to the row that made it | `RETURNING` works with `execute`, which gives back one row's id at a time |
| one `INSERT` with many `VALUES` and `RETURNING` | a batch small enough for one statement, whose ids you match by a column you sent | one statement returns every id, in no particular order, and it is capped by the placeholders a statement may hold |

The default is `executemany` inside `with conn:`, with the ids read back by a unique column when
they are needed. A commit after every row, whether written as `autocommit=True` or as a `commit` in
a loop, belongs to none of them.

### A loader for a day of readings

The pieces of this notebook in one loader: a station and a day of its readings from CSV text, with
corrections, in one transaction. An upsert with `RETURNING` gives the station's id whether the
station is new or not. `executemany` inserts the readings from a generator over the CSV, skipping
hours already loaded. A second `executemany` applies the corrections, and when `rowcount` shows one
matched nothing, the loader raises, and `with conn:` rolls everything back:


In [9]:
def load_day(conn, station, latitude, csv_text, corrections):
    """Load one station's day of readings and its corrections in one transaction, or change nothing."""
    with conn:
        station_id = conn.execute("""
            INSERT INTO stations (name, latitude) VALUES (?, ?)
            ON CONFLICT (name) DO UPDATE SET latitude = excluded.latitude
            RETURNING id
        """, (station, latitude)).fetchone()[0]
        rows = ((station_id, line["hour"], float(line["celsius"]) if line["celsius"] else None)
                for line in csv.DictReader(io.StringIO(csv_text)))
        inserted = conn.executemany(INSERT_READING + " ON CONFLICT (station_id, hour) DO NOTHING", rows).rowcount
        corrected = conn.executemany("UPDATE readings SET celsius = ? WHERE station_id = ? AND hour = ?",
                                     [(celsius, station_id, hour) for hour, celsius in corrections]).rowcount
        if corrected != len(corrections):
            raise ValueError(f"{len(corrections) - corrected} of {len(corrections)} corrections matched no reading")
    return station_id, inserted, corrected


BODO_DAY = "hour,celsius\n" + "".join(
    f"2026-01-05T{hour:02d}:00,{'' if hour == 7 else round(-4.0 + 0.3 * hour, 1)}\n" for hour in range(24))
reading = "SELECT celsius FROM readings WHERE station_id = ? AND hour = '2026-01-05T07:00'"

station_id, inserted, corrected = load_day(conn, "Bodo", 67.28, BODO_DAY, [("2026-01-05T07:00", -2.4)])
print(f"first run:  station {station_id}, {inserted} readings inserted, {corrected} corrected")
station_id, inserted, corrected = load_day(conn, "Bodo", 67.28, BODO_DAY, [("2026-01-05T07:00", -2.4)])
print(f"second run: station {station_id}, {inserted} readings inserted, {corrected} corrected")
try:
    load_day(conn, "Bodo", 67.28, BODO_DAY, [("2026-01-05T07:00", -2.6), ("2026-01-05 08:00", -1.5)])
except ValueError as error:
    print("third run refused:", error)
print("Bodo at 07:00:", conn.execute(reading, (station_id,)).fetchone()[0])


first run:  station 6, 24 readings inserted, 1 corrected
second run: station 6, 0 readings inserted, 1 corrected
third run refused: 1 of 2 corrections matched no reading
Bodo at 07:00: -2.4


The first run inserted Bodo's 24 hours, including the empty reading at 07:00, and filled that hour
in. The second run found the station and every hour already there, inserted nothing, and applied the
same correction again. The third run's second correction wrote its hour with a space, matched
nothing, and the whole run rolled back, so 07:00 still reads -2.4, not the -2.6 the run's first
correction wrote.

### Where each part came from

| In the loader | What it relies on | The section that showed it |
|---|---|---|
| `with conn:` around every statement | one transaction, rolled back when anything raises | A row that fails partway |
| `executemany(... ON CONFLICT ... DO NOTHING, rows)` | one call for every reading, fed by a generator | A generator in place of a list |
| `.rowcount` of the insert | the number of readings inserted, skipped ones not counted | A year of readings, in one call |
| `executemany("UPDATE ...", corrections)` | an `UPDATE` run for every item | UPDATE and DELETE, with dictionaries |
| `corrected != len(corrections)` | `rowcount` as a total, compared with what was sent | rowcount and lastrowid |
| `RETURNING id` on one `execute` | an id without `lastrowid` | Ids for many rows |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/14-executemany-solutions.ipynb).

**1.** Insert three new stations with one `executemany` call, from a list of dictionaries and named
placeholders, and print `rowcount` and the three stations' ids, read back by name.


In [10]:
# your code here


**2.** Delete every one of Tromso's readings from December 2025 with one `executemany` call that
deletes a day at a time, and print `rowcount`.


In [11]:
# your code here


**3.** Write a generator function that yields Svalbard's 24 readings for 1 January 2026, all -15.0,
load them with `executemany` inside `with conn:`, and print `rowcount` and `lastrowid`.


In [12]:
# your code here


**4.** Insert two stations with a loop of `execute` and `RETURNING id`, collecting a dictionary of
name to id, then use it to insert one reading for each station with `executemany`.


In [13]:
# your code here


**5.** Using `timed_load`, time the first 500 rows of `ROWS` with a commit for every row and with one
commit, and print whether the first took more than 20 times as long.


In [14]:
# your code here


**6.** Write a function that applies a list of `(station_id, hour, celsius)` corrections with
`executemany` inside `with conn:`, and raises when `rowcount` says a correction matched nothing, so
the others roll back. Test it with one correction for an hour that exists and one for an hour that
does not.


In [15]:
# your code here


## Common errors

### sqlite3.ProgrammingError: Incorrect number of bindings supplied. The current statement uses 1, and there are 16 supplied.


In [16]:
hours = ["2026-01-05T00:00", "2026-01-05T01:00"]
conn.executemany("DELETE FROM readings WHERE hour = ?", hours)


ProgrammingError: Incorrect number of bindings supplied. The current statement uses 1, and there are 16 supplied.

`executemany` treats every item as the values of one row, and a string is a sequence too: of its
characters. `"2026-01-05T00:00"` has 16 of them, so SQLite was given 16 values for one placeholder.
Every row needs to be a tuple, even a row of one value, which needs its comma:


In [17]:
with conn:
    deleted = conn.executemany("DELETE FROM readings WHERE hour = ?", [(hour,) for hour in hours])
print("deleted:", deleted.rowcount)


deleted: 2


### sqlite3.ProgrammingError: executemany() can only execute DML statements.


In [18]:
conn.executemany("SELECT COUNT(*) FROM readings WHERE station_id = ?", [(ids["Oslo"],), (ids["Tromso"],)])


ProgrammingError: executemany() can only execute DML statements.

`executemany` runs statements that change rows, and a `SELECT` changes none, so it refused before
running anything. Its rows would have had nowhere to go anyway: a cursor holds the rows of one query.
One query can answer for every station at once:


In [19]:
print(conn.execute("""
    SELECT station_id, COUNT(*) FROM readings WHERE station_id IN (?, ?) GROUP BY station_id ORDER BY station_id
""", (ids["Oslo"], ids["Tromso"])).fetchall())


[(2, 8760), (4, 8760)]


### sqlite3.InterfaceError: bad parameter or other API misuse


In [20]:
try:
    cursor = conn.executemany("INSERT INTO stations (name, latitude) VALUES (?, ?) RETURNING id",
                              [("Alesund", 62.47), ("Molde", 62.74)])
    print("no error, and the ids returned:", cursor.fetchall())
except sqlite3.Error as error:
    print(f"{type(error).__name__}: {error}")

inserted = conn.execute("SELECT COUNT(*) FROM stations WHERE name IN ('Alesund', 'Molde')").fetchone()[0]
print("stations inserted before it stopped:", inserted)
conn.rollback()


InterfaceError: bad parameter or other API misuse
stations inserted before it stopped: 1


Python's documentation says that `executemany` discards the rows a `RETURNING` clause produces. The
Python this notebook was written with, 3.14, instead inserts the first row and raises this error at
the second, so the cell prints whichever happens where it runs. Either way `executemany` gives no ids
back, and whatever it inserted waits in the transaction, which the rollback removed. For ids, use
`execute` with `RETURNING` for every row, or one `INSERT` with many `VALUES`, as the section on ids
showed:


In [21]:
with conn:
    added = {name: conn.execute("INSERT INTO stations (name, latitude) VALUES (?, ?) RETURNING id",
                                (name, latitude)).fetchone()[0]
             for name, latitude in [("Alesund", 62.47), ("Molde", 62.74)]}
print(added)


{'Alesund': 12, 'Molde': 13}


### No error, and Lakselv's reading filed under Kautokeino: lastrowid after executemany


In [22]:
cursor = conn.cursor()
cursor.execute("INSERT INTO stations (name, latitude) VALUES (?, ?)", ("Kautokeino", 69.01))
cursor.executemany("INSERT INTO stations (name, latitude) VALUES (?, ?)", [("Karasjok", 69.47), ("Lakselv", 70.05)])
lakselv = cursor.lastrowid                                           # meant to be the id of the last station
cursor.execute(INSERT_READING, (lakselv, "2026-01-05T12:00", -9.5))

NEWEST = "SELECT stations.name FROM readings JOIN stations ON stations.id = readings.station_id ORDER BY readings.id DESC"
print("the new reading belongs to:", conn.execute(NEWEST).fetchone()[0])
conn.rollback()


the new reading belongs to: Kautokeino


The reading meant for Lakselv went to Kautokeino. `executemany` inserted Karasjok and Lakselv without
touching `lastrowid`, which still held the id from the `execute` before it, a real id, of the wrong
station, so nothing failed. On a new cursor the same code would have passed `None`, and the insert
would have failed on `NOT NULL`. Read the ids back by name instead:


In [23]:
with conn:
    conn.execute("INSERT INTO stations (name, latitude) VALUES (?, ?)", ("Kautokeino", 69.01))
    conn.executemany("INSERT INTO stations (name, latitude) VALUES (?, ?)", [("Karasjok", 69.47), ("Lakselv", 70.05)])
    lakselv = conn.execute("SELECT id FROM stations WHERE name = ?", ("Lakselv",)).fetchone()[0]
    conn.execute(INSERT_READING, (lakselv, "2026-01-05T12:00", -9.5))

print("the new reading belongs to:", conn.execute(NEWEST).fetchone()[0])


the new reading belongs to: Lakselv


### No error, and a correction lost: rowcount after executemany is a total


In [24]:
corrections = [(-7.9, ids["Tromso"], "2025-12-24T18:00"), (-8.3, ids["Tromso"], "2025-12-24 19:00"),
               (-8.8, ids["Tromso"], "2025-12-24T20:00")]
with conn:
    cursor = conn.executemany("UPDATE readings SET celsius = ? WHERE station_id = ? AND hour = ?", corrections)

if cursor.rowcount:
    print("corrections applied, rowcount", cursor.rowcount)


corrections applied, rowcount 2


Three corrections went in and `rowcount` was 2, which the check read as success, since 2 is true.
The second correction's hour has a space where the `T` belongs, so it matched no reading and left the
reading it was meant to fix as it was. Nothing said so, because `rowcount` is the sum of every run of
the statement, and never says which run changed nothing. Compare the total with the number sent, and
roll back when they differ:


In [25]:
try:
    with conn:
        cursor = conn.executemany("UPDATE readings SET celsius = ? WHERE station_id = ? AND hour = ?", corrections)
        if cursor.rowcount != len(corrections):
            raise ValueError(f"{len(corrections) - cursor.rowcount} of {len(corrections)} corrections matched no reading")
except ValueError as error:
    print("rolled back:", error)


rolled back: 1 of 3 corrections matched no reading


### sqlite3.IntegrityError: UNIQUE constraint failed: readings.station_id, readings.hour


In [26]:
hourly = sqlite3.connect(DATABASE, autocommit=True)
day = [(ids["Kirkenes"], f"2026-01-02T{hour:02d}:00", -9.0 + hour / 4) for hour in range(24)]
day[15] = (ids["Kirkenes"], "2026-01-02T15:00", 999.0)
try:
    hourly.executemany(INSERT_READING, day)
except sqlite3.IntegrityError as error:
    print("first try refused:", error)

day[15] = (ids["Kirkenes"], "2026-01-02T15:00", -5.25)
hourly.executemany(INSERT_READING, day)


first try refused: CHECK constraint failed: plausible_celsius


IntegrityError: UNIQUE constraint failed: readings.station_id, readings.hour

The first try stopped at the reading of 999. With that reading fixed, the second try failed at its
very first row. With `autocommit=True`, every row was saved as `executemany` inserted it, so the 15
hours before the bad reading were already in the table, and the retry ran into the first of them.
There was no transaction for a rollback to undo. Remove what the failed load left, and load the day
in one transaction, which saves every hour or none:


In [27]:
leftover = hourly.execute("DELETE FROM readings WHERE station_id = ? AND hour LIKE '2026-01-02%'", (ids["Kirkenes"],))
print("rows the first try left:", leftover.rowcount)
hourly.close()

with conn:
    loaded = conn.executemany(INSERT_READING, day)
print("loaded in one transaction:", loaded.rowcount)


rows the first try left: 15
loaded in one transaction: 24


### No error, and nothing loaded: a generator checked before executemany


In [28]:
day = ((ids["Oslo"], f"2026-01-03T{hour:02d}:00", -2.0 + hour / 8) for hour in range(24))

missing = [row for row in day if row[2] is None]
print("readings missing a temperature:", len(missing))
with conn:
    loaded = conn.executemany(INSERT_READING, day)
print("loaded:", loaded.rowcount)


readings missing a temperature: 0
loaded: 0


The check found nothing missing, and the load inserted nothing. A generator can be read only once,
and the check read all of it, so `executemany` was handed a generator with nothing left, and
inserted nothing without an error. When rows have to be looked at twice, build a list, or make the
check part of the one pass:


In [29]:
day = [(ids["Oslo"], f"2026-01-03T{hour:02d}:00", -2.0 + hour / 8) for hour in range(24)]

missing = [row for row in day if row[2] is None]
print("readings missing a temperature:", len(missing))
with conn:
    loaded = conn.executemany(INSERT_READING, day)
print("loaded:", loaded.rowcount)
conn.close()


readings missing a temperature: 0
loaded: 24


Last, the connection is closed, so this cell removes the scratch folder, with both databases in it:


In [30]:
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


## Recap

- `executemany` runs one statement that changes rows once for every item of an iterable, a list or a
  generator, and every item is a tuple or a dictionary of one row's values, a one-value row included.
- Under the default transaction control every row joins one transaction, and a commit after every row
  takes far longer, because every commit waits for the disk.
- A generator feeds `executemany` one row at a time, and can be read only once.
- `rowcount` after `executemany` is the total of rows changed: compare it with the number of rows
  sent.
- `lastrowid` is not updated by `executemany`, so it is `None` on a new cursor and stale on a used
  one. Read ids back by a unique column, or use `RETURNING` with `execute`.
- `executemany` accepts no `SELECT`, and gives back no rows from `RETURNING`.
- A row that fails leaves the rows before it in the open transaction, so load inside `with conn:`,
  and never with `autocommit=True`.


## What is next

The **Indexes and Query Plans** notebook makes questions about all those rows fast: the plan SQLite
reports for a query with `EXPLAIN QUERY PLAN`, a `SCAN` of every row against a `SEARCH` through an
index, and the `WHERE` clause an index cannot help.


---

&#8592; **Previous:** [Changing a Schema](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/13-changing-a-schema.ipynb)  &nbsp;·&nbsp;  [sqlite3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Indexes and Query Plans](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/15-indexes-and-query-plans.ipynb) &#8594;
